<a href="https://colab.research.google.com/github/MaryamAlhosani1/TPCDS_Parquet_ORC/blob/main/TPCDS_Parquet_ORC_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

sf1 = pd.read_csv(
    "/content/TPCDS_Parquet_ORC/results/compression_sf1/compression_results.csv"
)

display(sf1[["table", "parquet_ratio", "orc_ratio", "winner"]].round(3))

,table,parquet_ratio,orc_ratio,winner
0,customer,0.424,0.409,ORC
1,item,0.356,0.341,ORC
2,store,3.313,1.725,ORC
3,date_dim,0.183,0.049,ORC
4,store_sales,0.380,0.444,Parquet


In [7]:
from pathlib import Path
import pandas as pd

results_dir = Path("/content/TPCDS_Parquet_ORC/results")

compression = pd.concat(
    [
        pd.read_csv(results_dir / f"compression_sf{scale}" / "compression_results.csv")
        for scale in (1, 2, 5)
    ],
    ignore_index=True,
)

display(compression.groupby("scale_factor").size().rename("tables"))

,tables
scale_factor,
1,5
2,5
5,5


In [8]:
size_columns = ["raw_bytes", "parquet_directory_bytes", "orc_directory_bytes"]
totals = compression.groupby("scale_factor")[size_columns].sum()

comparison = pd.DataFrame({
    "Parquet (% of raw size)": 100 * totals["parquet_directory_bytes"] / totals["raw_bytes"],
    "ORC (% of raw size)": 100 * totals["orc_directory_bytes"] / totals["raw_bytes"],
})

display(comparison.round(2))

,Parquet (% of raw size),ORC (% of raw size)
scale_factor,,
1,37.64,43.17
2,38.07,43.45
5,37.85,44.16


In [9]:
table_winners = compression.pivot(
    index="table",
    columns="scale_factor",
    values="winner",
).rename(columns=lambda scale: f"SF{scale}")

display(table_winners)

scale_factor,SF1,SF2,SF5
table,,,
customer,ORC,ORC,ORC
date_dim,ORC,ORC,ORC
item,ORC,ORC,ORC
store,ORC,ORC,ORC
store_sales,Parquet,Parquet,Parquet


Across SF1, SF2, and SF5, Parquet used less total storage than ORC across the five benchmark tables. At each scale, ORC produced smaller files for customer, item, store, and date_dim, while Parquet produced smaller files for store_sales.


In [10]:
from pathlib import Path
import pandas as pd

stats_dir = Path("/content/TPCDS_Parquet_ORC/results/statistics")
timing = pd.read_csv(stats_dir / "performance_summary.csv")

display(timing[[
    "scale_factor",
    "parquet_total_write_sec",
    "orc_total_write_sec",
    "parquet_total_count_sec",
    "orc_total_count_sec",
]].round(2))

,scale_factor,parquet_total_write_sec,orc_total_write_sec,parquet_total_count_sec,orc_total_count_sec
0,1,33.07,27.27,1.96,1.57
1,2,54.72,57.05,1.98,1.88
2,5,139.96,125.40,1.75,1.19


This cell loads the saved timing summary and compares the total write and row-count times across the five benchmark tables at each scale. ORC wrote faster at SF1 and SF5, while Parquet wrote faster at SF2. ORC completed the row-count operation faster at all three scales.

In [11]:
types = pd.read_csv(stats_dir / "data_type_comparison.csv")

display(types[[
    "data_type",
    "parquet_data_bytes",
    "orc_data_bytes",
    "observed_smaller_format",
    "paper_smaller_format",
]])

,data_type,parquet_data_bytes,orc_data_bytes,observed_smaller_format,paper_smaller_format
0,Date,296843,5737,ORC,Not separately reported
1,FloatingPoint,119386692,148346251,Parquet,Parquet
2,Integer,30687837,25027579,ORC,ORC
3,String,5352992,5235930,ORC,Parquet


This cell compares the compressed sizes of Parquet and ORC by data type at SF1. The saved measurements agreed with the original study for integers and floating-point values. ORC used less space for integers, and Parquet used less space for floating-point values. For strings, ORC used less space in these measurements, while the original study reported Parquet as smaller.

In [12]:
sensitivity = pd.read_csv(stats_dir / "sensitivity_analysis.csv")

data_files = sensitivity.loc[
    sensitivity["measure"].eq("data_bytes"),
    ["format", "before_bytes", "after_bytes", "reduction_pct"],
]

display(data_files.round(4))

,format,before_bytes,after_bytes,reduction_pct
0,parquet,5553790,5553068,0.0130
2,orc,5367091,5366848,0.0045


This cell compares the compressed data-file sizes before and after removing the final column from the SF1 customer table. The size decreased by about 0.0130% for Parquet and 0.0045% for ORC, so the effect was small for both formats in this test.